### Exercise 2: The "Human-in-the-Loop" Tool

Topic: Tool Calling & Manual Approval


Goal: Intercept an agent's decision to call a sensitive tool (Money Transfer).


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.messages import HumanMessage, ToolMessage

model = init_chat_model("groq:openai/gpt-oss-120b")

# Define the transfer_money tool
@tool
def transfer_money(amount: float, target_account: str):
    """Transfers money to a specific account."""
    return f"Successfully transferred ${amount} to {target_account}."

# Bind the tool to the model
llm_with_tools = model.bind_tools([transfer_money])

# Define the agent function that handles the conversation loop with human-in-the-loop approval
def run_agent(text: str):
    # Initialize messages with the user's input
    messages = [HumanMessage(content=text)]
    while True:
        # Invoke the model with the current conversation history
        response = llm_with_tools.invoke(messages)
        if not response.tool_calls:
            # If there are no tool calls, the model has generated a final response
            print('Final response:', response.content)
            print('\n\n', messages)
            break
        # Process each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            args = tool_call["args"]
            if tool_name == 'transfer_money':
                # Human-in-the-loop: Ask for confirmation before executing tool
                confirm = input(
                    f"Confirm transfer of ${args['amount']} to {args['target_account']}? (y/n): "
                )

                if confirm.lower() != 'y':
                    # If not confirmed, cancel the transfer and notify the model
                    print("Transfer cancelled.")
                    messages.append(response)
                    messages.append(
                        ToolMessage(
                            content='user cancelled the transfer.',
                            tool_call_id=tool_call["id"]
                        )
                    )
                    continue
                # If confirmed, execute the tool
                result = transfer_money.invoke(args)
                messages.append(response)
                messages.append(
                    ToolMessage(
                        content=result,
                        tool_call_id=tool_call["id"]
                    )
                )

# Run the agent with a sample transfer request
run_agent("I want to transfer $1000 to account 12345.")

Transfer cancelled.
Final response: The transfer has been cancelled as you requested. If you’d like to try again or need anything else, just let me know!


 [HumanMessage(content='I want to transfer $1000 to account 12345.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to transfer money. We have a function transfer_money. We should call it with amount 1000 and target_account "12345".', 'tool_calls': [{'id': 'fc_b8a317a8-6ffd-44d5-a9c5-22e9ea5c56d6', 'function': {'arguments': '{"amount":1000,"target_account":"12345"}', 'name': 'transfer_money'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 138, 'total_tokens': 206, 'completion_time': 0.178466084, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.00570677, 'prompt_tokens_details': None, 'queue_time': 0.04664006, 'total_time': 0.184172854}, 'model_name': 'openai/gpt-oss-120b', 'system_f